In [ ]:
import kagglehub
from pycocotools.coco import COCO
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

path = kagglehub.dataset_download(
    "awsaf49/coco-2017-dataset",
    "coco2017/annotations/instances_val2017.json"
)

# Initialize COCO API
coco = COCO(path)

In [ ]:
# Get category IDs to fetch subsets
category_ids = coco.getCatIds(catNms=['person', 'car', 'bicycle', 'truck', 'bus'])

# Get image Ids for these categories
# image_ids = coco.getImgIds(catIds=category_ids)
# Get all image IDs that contain any of our target classes
image_ids = set()
for category_id in category_ids:
    class_ids = coco.getImgIds(catIds=[category_id])
    image_ids.update(class_ids)
image_ids = list(image_ids)
print(len(image_ids))

# Get annotation Ids for these categories
ann_ids = coco.getAnnIds(imgIds=image_ids, catIds=category_ids)
print(len(ann_ids))

# Get categories
categories = coco.loadCats(category_ids)

In [ ]:
# Load data into dataframes
images_df = pd.DataFrame(coco.loadImgs(image_ids)).rename(columns={'id': 'image_id'})
ann_df = pd.DataFrame(coco.loadAnns(ann_ids))
category_df = pd.DataFrame(categories).rename(columns={'id': 'category_id'})

In [ ]:
# Link annotations to category names
merged = pd.merge(ann_df, category_df[['category_id', 'name', 'supercategory']], on='category_id')
# Link annotations to image metadata
final = pd.merge(merged, images_df, on='image_id')

final.head()

In [ ]:
final.shape

In [ ]:
# Remove null rows
cleaned_df = final.dropna(how='any')
cleaned_df.shape

In [ ]:
# Map COCO-original indices to 0-index for cleaner processing
map_ids = {old_id: index for index, old_id in enumerate(category_ids)}

# Create new column in dataframe to include 0-based indexing
cleaned_df['image_index'] = cleaned_df['category_id'].map(map_ids)

In [ ]:
cleaned_df.head()

In [ ]:
# Rearrange to make the index column to the 0th index

cols = cleaned_df.columns.tolist()
# Remove index column from current index and insert at 0th index
cols.insert(0, cols.pop(cols.index('image_index')))
# Reassign the dataframe with the new column order
cleaned_df = cleaned_df[cols]

In [ ]:
cleaned_df.head()

In [ ]:
# Delete columns
delete_columns = ['iscrowd']
cleaned_df.drop(columns=delete_columns, inplace=True)

In [ ]:
# One-hot Encoding
categorical_features = ['name']
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)], remainder='passthrough', verbose_feature_names_out=False)

# Fit and transform the data
encoded_df = ct.fit_transform(cleaned_df)

# Convert back to dataframe with column names
feature_names = ct.get_feature_names_out()
cleaned_df = pd.DataFrame(encoded_df, columns=feature_names)


cleaned_df.head()


In [ ]:
cleaned_df.shape

In [ ]:
# Convert dataframe to CSV and save to Kaggle's working directory
cleaned_df.to_csv('/kaggle/working/filtered_test_coco_metadata.csv', index=False)